In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, dynamic_prompt


# 1. Define runtime preferences
@dataclass
class UserContext:
    user_name: str
    language: str  # e.g., "Spanish", "English", "Japanese"
    country: str  # e.g., "US", "ES", "JP"


# 2. Dynamic prompt middleware that adapts instructions based on preferences
@dynamic_prompt
def localize_prompt(request: ModelRequest) -> str:
    # Extract runtime context passed during .invoke()
    ctx: UserContext = request.runtime.context

    return f"""
    You are a friendly customer service assistant.
    Current User: {ctx.user_name}
    
    INSTRUCTIONS:
    - Respond strictly in the language: {ctx.language}.
    - Format prices, dates, and regions for the country: {ctx.country}.
    """


# 3. Create agent (No tools needed to demonstrate dynamic prompt behavior)
agent = create_agent(
    model=mymodel, middleware=[localize_prompt], context_schema=UserContext
)

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "How much does the subscription cost?"}
        ]
    },
    context=UserContext(user_name="Carlos", language="German", country="Germnay"),
)

print(result["messages"][-1].content)

Response:

Hallo Carlos,

vielen Dank für Ihre Frage!

Damit ich Ihnen den genauen Preis nennen kann, benötige ich noch ein paar Informationen von Ihnen.

Um welche Art von Abonnement handelt es sich denn genau? Geht es um:

*   Ein **Streaming-Dienst** (z.B. Musik, Filme)?
*   Eine **Zeitschrift oder Zeitung**?
*   Eine **Mitgliedschaft** in einem Verein oder Studio?
*   Etwas ganz anderes?

Sobald ich mehr weiß, kann ich Ihnen gerne die genauen Kosten nennen.

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "How much does the subscription cost?"}
        ]
    },
    context=UserContext(user_name="Shyam", language="Hindi", country="India"),
)

print(result["messages"][-1].content)

Response:

नमस्ते श्याम,

यह जानने के लिए कि सदस्यता की लागत क्या है, कृपया मुझे बताएं कि आप किस विशेष सदस्यता के बारे में पूछ रहे हैं। क्या यह किसी सेवा, उत्पाद, या किसी अन्य चीज़ के लिए है?

एक बार जब आप मुझे अधिक जानकारी देंगे, तो मैं आपको सटीक लागत बता पाऊंगा।